# PneumoScore — kiểm tra bản vá trên ensemble cũ
Không huấn luyện lại. Add Input Dataset_phổi, Output Stability 3 Seeds và Output Ensemble Export Verify T4. GPU T4, Internet OFF. Giữ50/50 và ngưỡng0.55. Smoke fixture không phải Test BTC.

In [ ]:
import types
MODULE_SOURCE='"""Kaggle inference and verification for the team\'s two trusted checkpoints."""\nfrom pathlib import Path\nimport os,io,json,time,hashlib,pickle,zipfile,platform,shutil,gc\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image,ImageOps\nimport torch\nimport torchvision\nfrom torch import nn\nfrom torch.utils.data import Dataset,DataLoader\nfrom torchvision import transforms as T\nfrom torchvision.models import resnet18\n\nVERSION=\'pneumoscore_ensemble_export_v1_1\'\nLEGACY_MODEL_SHA256=\'d8055289142df2493d3f0e573e3e1420d2626ba7e7b64b8d05a14646497c9032\'\nPREPROCESS_ID=\'exif_gray_pad_bilinear_rgb_normalize_0.5_v1\'\nLABEL_NAMES={0:\'Normal\',1:\'Pneumonia\'}\nIMAGE_EXTENSIONS={\'.jpg\',\'.jpeg\',\'.png\',\'.bmp\',\'.tif\',\'.tiff\',\'.webp\'}\n\ndef ensure(condition,message):\n    if not condition:raise ValueError(message)\n\nclass BuiltinsOnlyUnpickler(pickle.Unpickler):\n    def find_class(self,module,name):\n        raise pickle.UnpicklingError(\'Globals forbidden in portable model: \'+module+\'.\'+name)\n\nclass KnownLegacyUnpickler(pickle.Unpickler):\n    # Used only AFTER the exact team\'s old file SHA256 has matched.\n    def find_class(self,module,name):\n        if module==\'numpy\' and name in {\'ndarray\',\'dtype\'}:return getattr(np,name)\n        if module in {\'numpy.core.multiarray\',\'numpy._core.multiarray\'} and name in {\'_reconstruct\',\'scalar\'}:\n            try:core=np._core.multiarray\n            except AttributeError:core=np.core.multiarray\n            return getattr(core,name)\n        raise pickle.UnpicklingError(\'Unsupported legacy global: \'+module+\'.\'+name)\n\ndef plain_value(value):\n    if isinstance(value,np.generic):return plain_value(value.item())\n    if value is None or type(value) in {str,bytes,int,float,bool}:return value\n    if isinstance(value,(list,tuple)):return [plain_value(x) for x in value]\n    if isinstance(value,dict):\n        ensure(all(type(k) is str for k in value),\'Metadata keys must be strings.\')\n        return {k:plain_value(v) for k,v in value.items()}\n    raise ValueError(\'Unsupported metadata value: \'+type(value).__name__)\n\ndef encode_portable(payload,source_sha256=None):\n    result=plain_value({k:v for k,v in payload.items() if k!=\'members\'})\n    result[\'export_schema\']=\'pneumoscore_probability_ensemble_bytes_v2\'\n    result[\'storage_version\']=2\n    if source_sha256:result[\'converted_from_sha256\']=source_sha256\n    result[\'members\']=[]\n    for member in payload[\'members\']:\n        tensors={}\n        for key,value in member[\'state_dict_numpy\'].items():\n            a=np.ascontiguousarray(value) if value.ndim else np.asarray(value).copy()\n            ensure(a.dtype.kind in \'fiu\' and not a.dtype.hasobject,\'Only numeric weight arrays allowed.\')\n            tensors[key]={\'dtype\':a.dtype.str,\'shape\':list(a.shape),\'data\':a.tobytes(order=\'C\')}\n        result[\'members\'].append({\'metadata\':plain_value(member[\'metadata\']),\'state_dict_bytes\':tensors})\n    return result\n\ndef decode_portable(raw):\n    ensure(raw.get(\'export_schema\')==\'pneumoscore_probability_ensemble_bytes_v2\',\'Unknown portable schema.\')\n    ensure(type(raw.get(\'members\')) is list and len(raw[\'members\'])==2,\'Expected two ensemble members.\')\n    result={k:v for k,v in raw.items() if k!=\'members\'};result[\'members\']=[]\n    for member in raw[\'members\']:\n        arrays={}\n        for key,item in member[\'state_dict_bytes\'].items():\n            dtype=np.dtype(item[\'dtype\']);shape=item[\'shape\'];data=item[\'data\']\n            ensure(dtype.kind in \'fiu\' and not dtype.hasobject and dtype.fields is None,\'Unsupported weight dtype.\')\n            ensure(type(shape) is list and all(type(x) is int and 0<=x<=100000 for x in shape) and len(shape)<=5,\'Invalid weight shape.\')\n            count=1\n            for dimension in shape:count*=dimension\n            ensure(type(data) is bytes and len(data)==count*dtype.itemsize,\'Invalid weight bytes length.\')\n            arrays[key]=np.frombuffer(data,dtype=dtype).reshape(tuple(shape)).copy()\n        result[\'members\'].append({\'metadata\':member[\'metadata\'],\'state_dict_numpy\':arrays})\n    return result\n\ndef read_model_payload(path):\n    path=Path(path);ensure(0<path.stat().st_size<=128*1024**2,\'Unexpected model file size.\')\n    digest=file_hash(path)\n    with path.open(\'rb\') as f:\n        try:raw=BuiltinsOnlyUnpickler(f).load()\n        except pickle.UnpicklingError:\n            ensure(digest==LEGACY_MODEL_SHA256,\'Legacy NumPy pickle is accepted only for the exact previously verified team file.\')\n            f.seek(0);raw=KnownLegacyUnpickler(f).load()\n    ensure(type(raw) is dict,\'Model payload must be a dictionary.\')\n    if raw.get(\'export_schema\')==\'pneumoscore_probability_ensemble_bytes_v2\':payload=decode_portable(raw)\n    else:\n        ensure(digest==LEGACY_MODEL_SHA256 and raw.get(\'export_schema\')==\'pneumoscore_probability_ensemble_numpy_v1\',\'Unknown legacy model.\')\n        payload=raw\n    ensure(payload.get(\'preprocess\')==PREPROCESS_ID and payload.get(\'image_size\')==224,\'Preprocessing mismatch.\')\n    ensure(payload.get(\'weights\')==[.5,.5] and payload.get(\'seed\')==2026 and payload.get(\'threshold\')==.55,\'The selected ensemble must remain fixed: seed2026, weights50/50, threshold0.55.\')\n    ensure(payload.get(\'label_mapping\')=={\'Normal\':0,\'Pneumonia\':1},\'Internal label mapping mismatch.\')\n    ensure(len(payload.get(\'members\',[]))==2,\'Expected two models.\')\n    return payload\n\ndef save_portable_pickle(payload,path,source_sha256=None):\n    raw=encode_portable(payload,source_sha256)\n    temp=Path(str(path)+\'.tmp\')\n    with temp.open(\'wb\') as f:pickle.dump(raw,f,protocol=4)\n    os.replace(temp,path)\n    with Path(path).open(\'rb\') as f:check=decode_portable(BuiltinsOnlyUnpickler(f).load())\n    for old,new in zip(payload[\'members\'],check[\'members\']):\n        ensure(old[\'state_dict_numpy\'].keys()==new[\'state_dict_numpy\'].keys(),\'Portable tensor keys changed.\')\n        for key,a in old[\'state_dict_numpy\'].items():\n            b=new[\'state_dict_numpy\'][key]\n            ensure(a.dtype==b.dtype and a.shape==b.shape and a.tobytes()==b.tobytes(),(\'Portable tensor changed\',key))\n    return check\n\ndef execution_environment():\n    import PIL\n    return dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,pillow=PIL.__version__,\n        torch=str(torch.__version__),torchvision=str(torchvision.__version__),cuda_available=torch.cuda.is_available(),\n        gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,precision=\'float32\',code_version=VERSION)\n\ndef require_kaggle(t4=False):\n    ensure(Path(\'/kaggle/input\').is_dir() and Path(\'/kaggle/working\').is_dir(),\'Run on Kaggle.\')\n    if t4:\n        ensure(torch.cuda.is_available(),\'Select GPU T4.\')\n        ensure(\'T4\' in torch.cuda.get_device_name(0),\'Verification requires T4.\')\n    torch.backends.cudnn.deterministic=True\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n\ndef save_json(value,path):\n    Path(path).write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False),encoding=\'utf-8\')\n\ndef file_hash(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for part in iter(lambda:f.read(1024*1024),b\'\'):h.update(part)\n    return h.hexdigest()\n\ndef manifest_fingerprint(frame):\n    cols=[\'relative_path\',\'label\',\'split\',\'sha256\',\'pixel_sha256\',\'patient_id\',\'group_id\']\n    content=frame.sort_values(\'relative_path\')[cols].to_csv(index=False,lineterminator=\'\\n\')\n    return hashlib.sha256(content.encode()).hexdigest()\n\ndef read_xray(path):\n    with Image.open(path) as source:\n        if source.mode in {\'I\',\'F\'} or source.mode.startswith(\'I;16\'):\n            raise ValueError(\'16-bit/float input needs a BTC-approved normalization policy; no automatic intensity conversion.\')\n        ensure(getattr(source,\'n_frames\',1)==1,\'Multi-frame image needs an explicit BTC-approved frame policy.\')\n        return ImageOps.exif_transpose(source).convert(\'L\')\n\nclass SquarePad:\n    def __init__(self,size):self.size=int(size)\n    def __call__(self,image):\n        return ImageOps.pad(image,(self.size,self.size),method=Image.Resampling.BILINEAR,color=0).convert(\'RGB\')\n\ndef inference_transform(size=224):\n    return T.Compose([SquarePad(size),T.ToTensor(),T.Normalize([.5]*3,[.5]*3)])\n\nclass ResNet18Baseline(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.backbone=resnet18(weights=None)\n        self.backbone.fc=nn.Linear(512,1)\n    def forward(self,x):\n        return self.backbone(x).squeeze(1)\n\nclass ECA(nn.Module):\n    def __init__(self, kernel_size=5):\n        super().__init__()\n        if kernel_size < 1 or kernel_size % 2 != 1:\n            raise ValueError(\'ECA kernel_size must be positive and odd.\')\n        self.pool = nn.AdaptiveAvgPool2d(1)\n        self.channel_filter = nn.Conv1d(1, 1, kernel_size, padding=kernel_size // 2, bias=False)\n    def forward(self, features):\n        descriptor = self.pool(features).flatten(1).unsqueeze(1)\n        gate = self.channel_filter(descriptor).squeeze(1).sigmoid()\n        return features * gate[:, :, None, None]\n\nclass ResNet18ECA(nn.Module):\n    def __init__(self, use_eca=True, kernel_size=5):\n        super().__init__()\n        self.backbone = resnet18(weights=None)\n        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, 1)\n        # Create ECA after the backbone and classifier, so a matched seed keeps them identical to the baseline.\n        self.eca = ECA(kernel_size) if use_eca else nn.Identity()\n        self.use_eca = bool(use_eca)\n        self.kernel_size = int(kernel_size)\n    def forward_features(self, x):\n        b = self.backbone\n        x = b.maxpool(b.relu(b.bn1(b.conv1(x))))\n        x = b.layer1(x)\n        x = b.layer2(x)\n        x = b.layer3(x)\n        return self.eca(b.layer4(x))\n    def forward(self, x):\n        features = self.forward_features(x)\n        pooled = self.backbone.avgpool(features).flatten(1)\n        return self.backbone.fc(pooled).squeeze(1)\n\ndef network_from_state(metadata,state):\n    arch=metadata[\'architecture\']\n    if arch==\'resnet18\':net=ResNet18Baseline()\n    elif arch==\'resnet18_eca\':net=ResNet18ECA(use_eca=True,kernel_size=int(metadata[\'eca_kernel\']))\n    else:raise ValueError(\'This bundle supports only the chosen baseline/ECA pair.\')\n    net.load_state_dict(state,strict=True)\n    return net\n\nclass ProbabilityEnsemble(nn.Module):\n    def __init__(self,members,weights):\n        super().__init__();self.members=nn.ModuleList(members)\n        ensure(len(weights)==len(members)==2 and all(w>0 for w in weights),\'Invalid ensemble member weights.\')\n        ensure(np.isclose(sum(weights),1.),\'Weights must sum to1.\')\n        self.weights=tuple(float(w) for w in weights)\n    def forward(self,images):\n        # This forward returns probabilities, so do not apply sigmoid a second time.\n        result=self.weights[0]*self.members[0](images).sigmoid()\n        return result+self.weights[1]*self.members[1](images).sigmoid()\n\ndef validate_member(saved,arch,seed,manifest_hash=None):\n    expected=dict(architecture=arch,seed=int(seed),pretrained=False,initialization=\'scratch\',\n        image_size=224,input_channels=3,label_mapping={\'Normal\':0,\'Pneumonia\':1},preprocess=PREPROCESS_ID)\n    if manifest_hash:expected[\'manifest_hash\']=manifest_hash\n    if arch==\'resnet18_eca\':expected[\'eca_kernel\']=5\n    for key,value in expected.items():ensure(saved.get(key)==value,(\'Checkpoint metadata mismatch\',key))\n    ensure(isinstance(saved.get(\'state_dict\'),dict) and saved.get(\'phase\')==\'main\',\'Invalid state/phase.\')\n    return saved\n\ndef find_run(arch,seed):\n    found={}\n    for p in Path(\'/kaggle/input\').rglob(\'run_config.json\'):\n        if not (p.parent/\'best.pt\').is_file():continue\n        cfg=json.loads(p.read_text())\n        if cfg.get(\'ARCH\')==arch and int(cfg.get(\'SEED\',-1))==seed:found[str(p.parent.resolve())]=p.parent\n    assert len(found)==1,(\'Attach the Stability notebook Output; expected one run\',arch,seed,len(found))\n    return next(iter(found.values()))\n\ndef find_selection(seed):\n    choices=list(Path(\'/kaggle/input\').rglob(\'selected_candidate.json\'))\n    assert len(choices)==1,\'Attach the saved Ensemble Error Analysis notebook Output.\'\n    chosen=json.loads(choices[0].read_text())\n    assert chosen[\'architectures\']==[\'resnet18\',\'resnet18_eca\'] and chosen[\'weights\']==[.5,.5]\n    threshold=float(chosen[\'per_seed_tuned_thresholds\'][str(seed)])\n    assert 0<threshold<1\n    return chosen,threshold,str(choices[0])\n\ndef resolve_data_root(config,manifest):\n    first=Path(str(manifest.iloc[0].relative_path))\n    assert not first.is_absolute() and \'..\' not in first.parts\n    configured=Path(config[\'DATA_ROOT\']) if config.get(\'DATA_ROOT\') else None\n    if configured and (configured/first).is_file():return configured\n    found={}\n    for p in Path(\'/kaggle/input\').rglob(first.name):\n        candidate=p\n        for _ in first.parts:candidate=candidate.parent\n        if (candidate/first).is_file():found[str(candidate.resolve())]=candidate\n    assert len(found)==1,\'Attach the BTC dataset; set DATA_ROOT if more than one copy is attached.\'\n    return next(iter(found.values()))\n\nclass InferenceDataset(Dataset):\n    def __init__(self,frame,root=None):\n        self.frame=frame.reset_index(drop=True).copy();self.root=root\n        self.transform=inference_transform()\n    def __len__(self):return len(self.frame)\n    def __getitem__(self,index):\n        r=self.frame.iloc[index]\n        path=Path(r.path) if self.root is None else self.root/str(r.relative_path)\n        image_id=str(r.Image_id) if self.root is None else str(r.image_id)\n        return self.transform(read_xray(path)),image_id\n\ndef loader_for(frame,config,root=None):\n    return DataLoader(InferenceDataset(frame,root),batch_size=int(config[\'BATCH_SIZE\']),shuffle=False,\n                      num_workers=int(config[\'NUM_WORKERS\']),pin_memory=torch.cuda.is_available(),\n                      persistent_workers=False,generator=torch.Generator().manual_seed(2027))\n\n@torch.inference_mode()\ndef predict(net,loader,device,probability_output=False):\n    net.eval();rows=[]\n    if device.type==\'cuda\':torch.cuda.synchronize()\n    start=time.perf_counter()\n    for images,ids in loader:\n        value=net(images.to(device,non_blocking=True))\n        p=value if probability_output else value.sigmoid()\n        ensure(p.ndim==1 and torch.isfinite(p).all() and ((p>=0)&(p<=1)).all(),\'Invalid predicted probabilities.\')\n        rows.extend({\'image_id\':str(i),\'p_pneumonia\':float(x)} for i,x in zip(ids,p.cpu().tolist()))\n    if device.type==\'cuda\':torch.cuda.synchronize()\n    return pd.DataFrame(rows),time.perf_counter()-start\n\ndef metrics(y,p,threshold):\n    pred=np.asarray(p)>=threshold;y=np.asarray(y,dtype=int)\n    tn=int(((y==0)&~pred).sum());fp=int(((y==0)&pred).sum())\n    fn=int(((y==1)&~pred).sum());tp=int(((y==1)&pred).sum())\n    a=(tn+tp)/len(y);se=tp/(tp+fn);sp=tn/(tn+fp);f1=2*tp/(2*tp+fp+fn)\n    return dict(threshold=float(threshold),Accuracy=a,Sensitivity=se,Specificity=sp,F1=f1,\n                P=.30*a+.25*se+.20*sp+.25*f1,TN=tn,FP=fp,FN=fn,TP=tp)\n\ndef compare_probabilities(actual,reference,threshold):\n    a=actual.sort_values(\'image_id\').reset_index(drop=True)\n    b=reference.sort_values(\'image_id\').reset_index(drop=True)\n    assert a.image_id.is_unique and a.image_id.tolist()==b.image_id.tolist(),\'Image IDs changed.\'\n    aa=a.p_pneumonia.to_numpy();bb=b.p_pneumonia.to_numpy()\n    assert np.allclose(aa,bb,atol=1e-6,rtol=1e-5),\'Probability reproduction failed.\'\n    assert np.array_equal(aa>=threshold,bb>=threshold),\'Prediction labels changed.\'\n    return dict(n_images=len(a),max_absolute_probability_error=float(np.max(np.abs(aa-bb))),\n                changed_labels=int(((aa>=threshold)!=(bb>=threshold)).sum()),atol=1e-6,rtol=1e-5)\n\ndef export_pickle(saved_members,weights,threshold,manifest,path,source_records):\n    keys=[\'schema\',\'code_version\',\'architecture\',\'eca_kernel\',\'seed\',\'pretrained\',\'initialization\',\n          \'image_size\',\'input_channels\',\'label_mapping\',\'preprocess\',\'manifest_hash\',\'initial_backbone_hash\',\n          \'phase\',\'best_epoch\',\'best_score\',\'best_loss\',\'threshold\']\n    payload=dict(export_schema=\'pneumoscore_probability_ensemble_numpy_v1\',version=VERSION,\n        architecture=\'resnet18_plus_resnet18_eca_probability_ensemble\',seed=2026,weights=list(weights),\n        threshold=float(threshold),threshold_origin=\'locked_existing_validation_selection_no_retuning\',\n        preprocess=PREPROCESS_ID,image_size=224,label_mapping={\'Normal\':0,\'Pneumonia\':1},\n        manifest_hash=saved_members[0][\'manifest_hash\'],source_checkpoints=source_records,\n        seen_hashes={\'file_sha256\':sorted(set(manifest.sha256)),\'pixel_sha256\':sorted(set(manifest.pixel_sha256))},\n        members=[dict(metadata={k:s[k] for k in keys if k in s},\n          state_dict_numpy={k:v.detach().cpu().contiguous().numpy().copy() for k,v in s[\'state_dict\'].items()}) for s in saved_members])\n    save_portable_pickle(payload,path)\n    return payload\n\ndef load_pickle(path,device):\n    payload=read_model_payload(path);nets=[]\n    for item,arch in zip(payload[\'members\'],[\'resnet18\',\'resnet18_eca\']):\n        state={k:torch.from_numpy(v.copy()) for k,v in item[\'state_dict_numpy\'].items()}\n        meta=dict(item[\'metadata\'],state_dict=state)\n        validate_member(meta,arch,payload[\'seed\'],payload[\'manifest_hash\'])\n        nets.append(network_from_state(meta,state))\n    ensure(nets[0].state_dict().keys()!=nets[1].state_dict().keys(),\'Wrong ensemble architectures.\')\n    return ProbabilityEnsemble(nets,payload[\'weights\']).to(device).eval(),payload\n\n@torch.inference_mode()\ndef benchmark_forward(nets,images,device):\n    records=[]\n    for batch in [1,len(images)]:\n        x=images[:batch].to(device)\n        for name,net in nets.items():\n            for _ in range(10):net(x)\n            torch.cuda.synchronize();times=[]\n            for _ in range(50):\n                begin=time.perf_counter();net(x);torch.cuda.synchronize()\n                times.append(1000*(time.perf_counter()-begin))\n            records.append(dict(model=name,batch_size=batch,warmup_batches=10,repetitions=50,\n                median_batch_ms=float(np.median(times)),p95_batch_ms=float(np.percentile(times,95)),\n                amortized_ms_per_image=float(np.median(times)/batch),\n                throughput_images_per_second=float(1000*batch/np.median(times)),\n                scope=\'FP32 forward only, input already on GPU, synchronous wall time; not disk/preprocessing\'))\n    return records\n\ndef validate_csv(frame,path):\n    ensure(list(frame.columns)==[\'Image_id\',\'Prediction\'] and len(frame)>0,\'CSV must contain Image_id,Prediction and at least one row.\')\n    ensure(frame.Image_id.is_unique and frame.Image_id.map(lambda x:isinstance(x,str) and bool(x.strip())).all(),\'CSV IDs must be unique nonempty strings.\')\n    ensure(set(frame.Prediction).issubset({\'Normal\',\'Pneumonia\'}),\'BTC Article6 requires Normal/Pneumonia strings, not internal0/1.\')\n    frame.to_csv(path,index=False)\n    loaded=pd.read_csv(path,dtype=str,keep_default_na=False)\n    ensure(loaded.equals(frame),\'CSV round-trip mismatch.\')\n\ndef make_submission_notebook(module_source):\n    config=dict(MODE=\'submit\',MODEL_PATH=\'\',TEST_DIR=\'\',SAMPLE_SUBMISSION_CSV=\'\',TEAM_ID=\'team01\',\n                BATCH_SIZE=32,NUM_WORKERS=2,OUTPUT=\'/kaggle/working/ensemble_submission\')\n    def code(s):return dict(cell_type=\'code\',metadata={},source=s,execution_count=None,outputs=[])\n    return dict(nbformat=4,nbformat_minor=4,metadata={\'kernelspec\':{\'name\':\'python3\',\'display_name\':\'Python 3\',\'language\':\'python\'},\n        \'language_info\':{\'name\':\'python\'},\'kaggle\':{\'isInternetEnabled\':False,\'isGpuEnabled\':True}},cells=[\n        dict(cell_type=\'markdown\',metadata={},source=\'# PneumoScore ensemble — suy luận Test\\nAdd Input model.pkl của đội và tập Test BTC. Điền TEST_DIR, mã đội và CSV mẫu nếu BTC cung cấp, rồi Run All trên Kaggle. Không cần nhãn Test, không huấn luyện lại, không sửa CSV thủ công. Trọng số ghép và ngưỡng nằm trong model.pkl.\\n\'),\n        code(\'CONFIG=\'+repr(config)+\'\\n\'),code(module_source),\n        code("CONFIG[\'_MODULE_SOURCE\']="+repr(module_source)+"\\nrun_submission(CONFIG)\\n")])\n\ndef make_verification_notebook(config,module_source):\n    def code(s):return dict(cell_type=\'code\',metadata={},source=s,execution_count=None,outputs=[])\n    return dict(nbformat=4,nbformat_minor=4,metadata={\'kernelspec\':{\'name\':\'python3\',\'display_name\':\'Python 3\',\'language\':\'python\'},\n        \'language_info\':{\'name\':\'python\'},\'kaggle\':{\'isInternetEnabled\':False,\'isGpuEnabled\':True}},cells=[\n        dict(cell_type=\'markdown\',metadata={},source=\'# Kiểm tra lại ensemble seed 2026\\nChạy trên Kaggle GPU T4. Add Input Output Stability 3 Seeds, Output Ensemble Error Analysis 3 Seeds và Dataset_phổi. Không train lại, không tải pretrained.\\n\'),\n        code(\'CONFIG=\'+repr(config)+\'\\n\'),code(module_source),\n        code(\'MODULE_SOURCE=\'+repr(module_source)+\'\\nverify_export(CONFIG,MODULE_SOURCE)\\n\')])\n\ndef resolve_model(config):\n    if config.get(\'MODEL_PATH\'):\n        p=Path(config[\'MODEL_PATH\']);ensure(p.is_file(),\'MODEL_PATH does not exist.\');return p\n    choices={str(p.resolve()):p for p in Path(\'/kaggle/input\').rglob(\'model.pkl\')}\n    ensure(len(choices)==1,\'Set MODEL_PATH explicitly if zero or multiple model.pkl files are attached.\')\n    return next(iter(choices.values()))\n\ndef collect_test(config,payload,out):\n    ensure(config.get(\'TEST_DIR\'),\'Set TEST_DIR to BTC Test images.\')\n    root=Path(config[\'TEST_DIR\']).resolve();mode=config.get(\'MODE\',\'submit\')\n    ensure(mode in {\'submit\',\'smoke\'},\'MODE must be submit or smoke.\')\n    allowed=Path(\'/kaggle/input\') if mode==\'submit\' else Path(\'/kaggle/working\')\n    ensure(root.is_dir() and root.is_relative_to(allowed.resolve()),\'Test directory outside the allowed Kaggle root for this mode.\')\n    visible=sorted(p for p in root.rglob(\'*\') if p.is_file() and not any(x.startswith(\'.\') for x in p.relative_to(root).parts))\n    unsupported=[p for p in visible if p.suffix.lower() in {\'.dcm\',\'.dicom\',\'.gif\',\'.jp2\',\'.pgm\',\'.pnm\'}]\n    files=[p for p in visible if p.suffix.lower() in IMAGE_EXTENSIONS]\n    ensure(files or unsupported,\'No image files found in TEST_DIR.\')\n    errors=[dict(Image_id=p.relative_to(root).as_posix(),path=str(p),error=\'Unsupported image format; obtain a BTC-approved reader/policy.\') for p in unsupported]\n    if not files:\n        pd.DataFrame(errors,columns=[\'Image_id\',\'path\',\'error\']).to_csv(out/\'input_errors.csv\',index=False)\n        raise ValueError(\'Only unsupported images found; see input_errors.csv.\')\n    def normalized(value):return str(value).replace(\'\\\\\',\'/\')\n    def identifier_stem(value):\n        p=Path(value)\n        return str(p.with_suffix(\'\')) if p.suffix.lower() in IMAGE_EXTENSIONS else str(p)\n    by_relative={p.relative_to(root).as_posix():p for p in files}\n    indexes={kind:{} for kind in [\'name\',\'relative_case\',\'name_case\',\'relative_stem\',\'stem_case\']}\n    for p in files:\n        rel=p.relative_to(root).as_posix()\n        for kind,key in [(\'name\',p.name),(\'relative_case\',rel.casefold()),(\'name_case\',p.name.casefold()),(\'relative_stem\',str(Path(rel).with_suffix(\'\')).casefold()),(\'stem_case\',p.stem.casefold())]:\n            indexes[kind].setdefault(key,[]).append(p)\n    mapping=[];paths=[];ids=[]\n    if config.get(\'SAMPLE_SUBMISSION_CSV\'):\n        sample=pd.read_csv(config[\'SAMPLE_SUBMISSION_CSV\'],usecols=[\'Image_id\'],dtype=str,keep_default_na=False)\n        ids=sample.Image_id.tolist()\n        ensure(ids and len(set(ids))==len(ids) and all(x.strip() for x in ids),\'Sample IDs must be unique and nonempty.\')\n        for original in ids:\n            name=normalized(original);selected=None;method=None\n            for kind,key in [(\'exact_relative\',name),(\'name\',name),(\'relative_case\',name.casefold()),(\'name_case\',name.casefold()),(\'relative_stem\',identifier_stem(name).casefold()),(\'stem_case\',Path(identifier_stem(name)).name.casefold())]:\n                options=[by_relative[key]] if kind==\'exact_relative\' and key in by_relative else [] if kind==\'exact_relative\' else indexes[kind].get(key,[])\n                if options:\n                    ensure(len(options)==1,(\'Ambiguous sample Image_id; no prediction guessed\',original,kind))\n                    selected=options[0];method=kind;break\n            ensure(selected is not None,(\'Sample ID has no matching image; no prediction guessed\',original))\n            paths.append(selected);mapping.append(dict(Image_id=original,relative_path=selected.relative_to(root).as_posix(),match_method=method))\n        ensure(len(set(paths))==len(paths),\'Multiple sample IDs resolve to the same physical image.\')\n    else:\n        ensure(all(len(v)==1 for v in indexes[\'name\'].values()),\'Duplicate basenames require an authoritative BTC sample CSV.\')\n        paths=files;ids=[p.name for p in paths]\n        mapping=[dict(Image_id=i,relative_path=p.relative_to(root).as_posix(),match_method=\'filename_without_sample\') for i,p in zip(ids,paths)]\n    pd.DataFrame(mapping).to_csv(out/\'test_id_mapping.csv\',index=False)\n    extras=[p.relative_to(root).as_posix() for p in files if p not in set(paths)]\n    save_json(dict(sample_is_authoritative=bool(config.get(\'SAMPLE_SUBMISSION_CSV\')),extra_image_paths=extras,unsupported_image_paths=[str(p.relative_to(root)) for p in unsupported]),out/\'input_scan_audit.json\')\n    if extras:print(\'WARNING sample IDs define output; extra images logged:\',len(extras),flush=True)\n    old_file=set(payload[\'seen_hashes\'][\'file_sha256\']);old_pixel=set(payload[\'seen_hashes\'][\'pixel_sha256\']);rows=[]\n    for image_id,p in zip(ids,paths):\n        try:\n            image=read_xray(p);a=file_hash(p);b=hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()\n            rows.append(dict(Image_id=image_id,path=str(p),sha256=a,pixel_sha256=b,overlap_file=a in old_file,overlap_pixel=b in old_pixel))\n        except Exception as error:errors.append(dict(Image_id=image_id,path=str(p),error=type(error).__name__+\': \'+str(error)))\n    pd.DataFrame(errors,columns=[\'Image_id\',\'path\',\'error\']).to_csv(out/\'input_errors.csv\',index=False)\n    ensure(not errors,(\'Unreadable/unsupported images: no majority-class fallback, no complete submission generated; see input_errors.csv\',len(errors)))\n    frame=pd.DataFrame(rows);frame.to_csv(out/\'test_file_audit.csv\',index=False)\n    overlap=int((frame.overlap_file|frame.overlap_pixel).sum())\n    if overlap:print(\'WARNING\',overlap,\'images overlap prior Train/Validation; predictions retained for every requested ID; BTC review required.\',flush=True)\n    return frame\n\ndef run_submission(config):\n    require_kaggle();out=Path(config[\'OUTPUT\']).resolve()\n    ensure(out.is_relative_to(Path(\'/kaggle/working\').resolve()),\'OUTPUT must be under /kaggle/working.\')\n    out.mkdir(parents=True,exist_ok=True);mode=config.get(\'MODE\',\'submit\')\n    team=config.get(\'TEAM_ID\',\'team01\');ensure(team and all(c.isalnum() or c in \'_-\' for c in team),\'Invalid team ID.\')\n    final_name=team+\'_submission.zip\' if mode==\'submit\' else \'SMOKE_ONLY_\'+team+\'.zip\'\n    # Archive previous generated outputs so a failed rerun cannot expose a stale final CSV/ZIP.\n    old=[out/name for name in [team+\'.csv\',final_name,\'submission.ipynb\',\'submission_metadata.json\'] if (out/name).is_file()]\n    if old:\n        backup=out/\'previous_runs\'/str(time.time_ns());backup.mkdir(parents=True)\n        for p in old:shutil.move(str(p),str(backup/p.name))\n    save_json(dict(status=\'RUNNING\',mode=mode),out/\'run_status.json\')\n    try:\n        device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        source=resolve_model(config);net,payload=load_pickle(source,device)\n        source_sha=file_hash(source);save_json(execution_environment(),out/\'environment.json\')\n        frame=collect_test(config,payload,out)\n        probabilities,_=predict(net,loader_for(frame,config),device,True)\n        ensure(probabilities.image_id.tolist()==frame.Image_id.tolist(),\'Prediction order differs from requested IDs.\')\n        labels=pd.DataFrame({\'Image_id\':probabilities.image_id,\'Prediction\':[LABEL_NAMES[int(x>=payload[\'threshold\'])] for x in probabilities.p_pneumonia]})\n        csv_path=out/(team+\'.csv\');validate_csv(labels,csv_path)\n        probabilities.rename(columns={\'image_id\':\'Image_id\'}).to_csv(out/\'test_probabilities_internal.csv\',index=False)\n        save_portable_pickle(payload,out/\'model.pkl\',payload.get(\'converted_from_sha256\',source_sha))\n        notebook=make_submission_notebook(config[\'_MODULE_SOURCE\'])\n        replay={k:v for k,v in config.items() if not k.startswith(\'_\')};replay[\'MODEL_PATH\']=\'\'\n        notebook[\'cells\'][1][\'source\']=\'CONFIG=\'+repr(replay)+\'\\n\'\n        for i,cell in enumerate(notebook[\'cells\']):\n            if cell[\'cell_type\']==\'code\':compile(\'\'.join(cell[\'source\']),\'submission_cell_\'+str(i),\'exec\')\n        save_json(notebook,out/\'submission.ipynb\')\n        overlap=int((frame.overlap_file|frame.overlap_pixel).sum())\n        metadata=dict(version=VERSION,mode=mode,purpose=\'official_test_inference\' if mode==\'submit\' else \'smoke_fixture_not_BTC_Test\',\n            seed=payload[\'seed\'],weights=payload[\'weights\'],threshold=payload[\'threshold\'],n_images=len(frame),\n            output_labels=[\'Normal\',\'Pneumonia\'],test_labels_used=False,overlap_images=overlap,\n            eligibility_review_required=bool(overlap) or mode!=\'submit\',independent_test_score_verified=False,\n            model_source_sha256=source_sha,model_sha256=file_hash(out/\'model.pkl\'),csv_sha256=file_hash(csv_path),environment=execution_environment())\n        save_json(metadata,out/\'submission_metadata.json\')\n        dest=out/final_name;temporary=Path(str(dest)+\'.tmp\')\n        with zipfile.ZipFile(temporary,\'w\',zipfile.ZIP_DEFLATED) as z:\n            for name in [\'model.pkl\',team+\'.csv\',\'submission.ipynb\',\'submission_metadata.json\',\'test_file_audit.csv\',\'test_id_mapping.csv\',\'input_scan_audit.json\',\'environment.json\']:z.write(out/name,name)\n        with zipfile.ZipFile(temporary) as z:ensure(z.testzip() is None,\'ZIP CRC failed.\')\n        os.replace(temporary,dest)\n        status=\'SMOKE_ONLY\' if mode==\'smoke\' else \'GENERATED_REVIEW_REQUIRED\' if overlap else \'GENERATED\'\n        save_json(dict(status=status,zip=str(dest),n_images=len(frame),eligibility_review_required=metadata[\'eligibility_review_required\']),out/\'run_status.json\')\n        print(status,str(dest),\'rows\',len(frame),\'threshold0.55 | no Test score claimed\',flush=True)\n        return dest\n    except Exception as error:\n        save_json(dict(status=\'FAILED\',mode=mode,error=type(error).__name__+\': \'+str(error),previous_outputs_archived=True),out/\'run_status.json\')\n        raise\n\ndef verify_export(config,module_source):\n    require_kaggle(t4=True);device=torch.device(\'cuda:0\');out=Path(config[\'OUTPUT\']);out.mkdir(parents=True,exist_ok=True)\n    seed=2026;selection,threshold,selection_source=find_selection(seed)\n    assert threshold==.55,\'The candidate threshold must remain locked to the previous analysis.\'\n    run_dirs=[find_run(a,seed) for a in [\'resnet18\',\'resnet18_eca\']]\n    manifests=[(d/\'split_manifest.csv\').read_bytes() for d in run_dirs]\n    assert manifests[0]==manifests[1]\n    manifest=pd.read_csv(io.BytesIO(manifests[0]),keep_default_na=False)\n    fingerprint=manifest_fingerprint(manifest)\n    assert fingerprint==\'9f9474fafb054f501535269fa3e55bea4c3e24d794f6afe9c019abe6ce798ac2\'\n    val=manifest[manifest.split==\'validation\'].sort_values(\'image_id\').reset_index(drop=True)\n    assert len(val)==1028 and val.image_id.is_unique and len(manifest)==5132\n    assert not set(manifest.loc[manifest.split==\'train\',\'pixel_sha256\'])&set(val.pixel_sha256)\n    saved=[];references=[];provenance=[];nets=[];source_metrics=[]\n    for arch,d in zip([\'resnet18\',\'resnet18_eca\'],run_dirs):\n        p=d/\'best.pt\';item=validate_member(torch.load(p,map_location=\'cpu\',weights_only=True),arch,seed,fingerprint)\n        saved.append(item);net=network_from_state(item,item[\'state_dict\']).to(device).eval();nets.append(net)\n        reference=pd.read_csv(d/\'val_predictions.csv\').sort_values(\'image_id\').reset_index(drop=True)\n        assert reference.image_id.tolist()==val.image_id.tolist() and np.array_equal(reference.true_label.to_numpy(),val.label.to_numpy())\n        references.append(reference);source_metrics.append(json.loads((d/\'metrics.json\').read_text()))\n        provenance.append(dict(architecture=arch,seed=seed,checkpoint=str(p),checkpoint_sha256=file_hash(p),\n            predictions_sha256=file_hash(d/\'val_predictions.csv\'),manifest_sha256=file_hash(d/\'split_manifest.csv\'),best_epoch=item[\'best_epoch\']))\n    assert saved[0][\'initial_backbone_hash\']==saved[1][\'initial_backbone_hash\']\n    root=resolve_data_root(config,manifest)\n    print(\'CHECKPOINT_PAIR_OK | seed2026 | baseline + ECA | weights 0.5/0.5 | threshold\',threshold,flush=True)\n    # Verify the actual BTC image bytes and normalized pixels against the original manifest.\n    for count,r in enumerate(val.itertuples(index=False),1):\n        rel=Path(str(r.relative_path));assert not rel.is_absolute() and \'..\' not in rel.parts\n        p=root/rel;assert file_hash(p)==r.sha256,\'Validation image bytes changed.\'\n        image=read_xray(p);assert hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()==r.pixel_sha256\n        if count%250==0:print(\'IMAGE_AUDIT\',count,\'/ 1028\',flush=True)\n    print(\'IMAGE_AUDIT_OK | 1028/1028 | original preprocessing\',flush=True)\n    environment=dict(torch=str(torch.__version__),torchvision=str(torchvision.__version__),python=platform.python_version(),\n                     device=str(device),gpu=torch.cuda.get_device_name(0),visible_gpu_count=torch.cuda.device_count(),\n                     used_gpu_indices=[0],precision=\'float32\',new_training=False,pretrained=False)\n    save_json(environment,out/\'environment.json\')\n    loader=loader_for(val,config,root);actual=[];checks={};metric_rows=[]\n    for arch,net,reference,sm in zip([\'resnet18\',\'resnet18_eca\'],nets,references,source_metrics):\n        frame,_=predict(net,loader,device)\n        checks[arch]=compare_probabilities(frame,reference,float(sm[\'validation_tuned\'][\'threshold\']))\n        actual.append(frame)\n        m=metrics(val.label,frame.p_pneumonia,float(sm[\'validation_tuned\'][\'threshold\']))\n        for k,v in sm[\'validation_tuned\'].items():\n            if k in m:assert np.isclose(m[k],v,atol=1e-12,rtol=0)\n        metric_rows.append(dict(model=arch,**m))\n        print(\'ORIGINAL_PROBABILITIES_MATCH\',arch,json.dumps(checks[arch]),flush=True)\n    model_path=out/\'model.pkl\'\n    payload=export_pickle(saved,[.5,.5],threshold,manifest,model_path,provenance)\n    loaded,payload=load_pickle(model_path,device)\n    reloaded,_=predict(loaded,loader,device,True)\n    expected=references[0][[\'image_id\',\'p_pneumonia\']].copy()\n    expected[\'p_pneumonia\']=(references[0].p_pneumonia.to_numpy()+references[1].p_pneumonia.to_numpy())/2\n    checks[\'ensemble_pickle_reload\']=compare_probabilities(reloaded,expected,threshold)\n    previous_tables=list(Path(\'/kaggle/input\').rglob(\'metrics_validation_tuned.csv\'))\n    assert len(previous_tables)==1,\'Attach the Analysis Output for the exact existing candidate comparison.\'\n    previous=pd.read_csv(previous_tables[0]);r=previous[(previous.method==selection[\'method\'])&(previous.seed==seed)]\n    assert len(r)==1\n    ensemble_metrics=metrics(val.label,reloaded.p_pneumonia,threshold)\n    for k,v in ensemble_metrics.items():assert np.isclose(v,float(r.iloc[0][k]),atol=1e-12,rtol=0),k\n    metric_rows.append(dict(model=\'ensemble_50_50\',**ensemble_metrics))\n    print(\'MODEL_PKL_FULL_VALIDATION_RELOAD_OK\',json.dumps(checks[\'ensemble_pickle_reload\']),flush=True)\n    detailed=val[[\'image_id\',\'label\']].copy()\n    for name,frame in zip([\'baseline_probability\',\'eca_probability\',\'ensemble_probability\'],actual+[reloaded]):detailed[name]=frame.p_pneumonia.to_numpy()\n    detailed[\'ensemble_prediction\']=[LABEL_NAMES[int(x>=threshold)] for x in reloaded.p_pneumonia]\n    detailed.to_csv(out/\'validation_predictions_reloaded.csv\',index=False,float_format=\'%.17g\')\n    schema=pd.DataFrame({\'Image_id\':reloaded.image_id,\'Prediction\':detailed.ensemble_prediction})\n    validate_csv(schema,out/\'validation_format_check.csv\')\n    pd.DataFrame(metric_rows).to_csv(out/\'validation_metrics.csv\',index=False)\n    batch,_=next(iter(loader));forward=benchmark_forward({\'resnet18\':loaded.members[0],\'ensemble_50_50\':loaded},batch,device)\n    e2e=[]\n    # Both have already processed the entire set. Alternate order across two measured passes.\n    for repeat,order in enumerate([[\'resnet18\',\'ensemble_50_50\'],[\'ensemble_50_50\',\'resnet18\']],1):\n        for name in order:\n            net=loaded.members[0] if name==\'resnet18\' else loaded\n            frame,seconds=predict(net,loader,device,name!=\'resnet18\')\n            compare_probabilities(frame,references[0] if name==\'resnet18\' else expected,\n                                  float(source_metrics[0][\'validation_tuned\'][\'threshold\']) if name==\'resnet18\' else threshold)\n            e2e.append(dict(model=name,repeat=repeat,n_images=len(frame),seconds=seconds,ms_per_image=1000*seconds/len(frame),\n                            images_per_second=len(frame)/seconds,scope=\'disk read + exif/grayscale/pad/normalize + DataLoader + H2D + forward + D2H; warmed cache, two alternating-order passes\'))\n    size=dict(model_pkl_bytes=model_path.stat().st_size,model_pkl_MiB=model_path.stat().st_size/1024**2,\n              baseline_parameters=sum(p.numel() for p in loaded.members[0].parameters()),\n              eca_parameters=sum(p.numel() for p in loaded.members[1].parameters()),\n              ensemble_parameters=sum(p.numel() for p in loaded.parameters()),\n              baseline_state_tensor_bytes=sum(v.numel()*v.element_size() for v in saved[0][\'state_dict\'].values()),\n              ensemble_state_tensor_bytes=sum(v.numel()*v.element_size() for s in saved for v in s[\'state_dict\'].values()))\n    assert size[\'ensemble_parameters\']==22354055\n    bench=dict(environment=environment,forward=forward,end_to_end=e2e,size=size)\n    save_json(bench,out/\'benchmark.json\');pd.DataFrame(forward).to_csv(out/\'benchmark_forward.csv\',index=False)\n    pd.DataFrame(e2e).to_csv(out/\'benchmark_end_to_end.csv\',index=False)\n    save_json(dict(status=\'passed\',candidate_selection_source=selection_source,seed=seed,weights=[.5,.5],threshold=threshold,\n        n_validation=len(val),image_bytes_and_pixels_match_manifest=True,checks=checks,\n        model_sha256=file_hash(model_path),no_new_training=True,test_labels_used=False,independent_test_verified=False),out/\'verification.json\')\n    save_json(provenance,out/\'source_checkpoints.json\')\n    (out/\'ensemble_inference.py\').write_text(module_source,encoding=\'utf-8\')\n    submission=make_submission_notebook(module_source)\n    save_json(submission,out/\'submission.ipynb\')\n    save_json(make_verification_notebook(config,module_source),out/\'verify_export.ipynb\')\n    report=[\'# Ensemble ResNet18 gốc + ECA seed 2026 đã kiểm tra trọng số\',\'\',\n      \'Cấu hình: ghép xác suất 50/50, ngưỡng 0,55 đã khóa từ phân tích validation trước. Dùng hai checkpoint cùng seed 2026, không train lại. Phiên kiểm tra dùng GPU Tesla T4, FP32, GPU số 0.\',\n      \'\', \'## Kiểm tra đã đạt\',\'\',\n      \'- Hai checkpoint đúng kiến trúc/seed, cùng manifest và backbone khởi tạo.\',\n      \'- Cả 1.028 ảnh validation khớp byte và pixel với manifest cũ.\',\n      \'- Xác suất từng model khớp CSV nguồn (atol 1e-6, rtol 1e-5), nhãn không thay đổi.\',\n      \'- model.pkl chứa cả hai bộ trọng số dạng NumPy và metadata. Đã nạp lại model.pkl và suy luận toàn bộ validation, khớp ensemble CSV và confusion matrix đã đo.\',\n      \'- CSV kiểm tra định dạng có đúng Image_id, Prediction. Đây là validation_format_check.csv, không phải bài nộp Test.\',\n      \'\', \'## Chỉ số validation của một cặp seed 2026\',\'\',\n      \'| Model | Ngưỡng | P (%) | Accuracy (%) | Sensitivity (%) | Specificity (%) | FN | FP |\',\n      \'|---|---:|---:|---:|---:|---:|---:|---:|\']\n    for r in metric_rows:report.append(f"| {r[\'model\']} | {r[\'threshold\']:.2f} | {100*r[\'P\']:.4f} | {100*r[\'Accuracy\']:.4f} | {100*r[\'Sensitivity\']:.4f} | {100*r[\'Specificity\']:.4f} | {r[\'FN\']} | {r[\'FP\']} |")\n    report+=[\'\',\'## Tốc độ và dung lượng\',\'\',\'| Model | Batch | Forward median (ms/batch) | P95 (ms/batch) | Images/s |\',\'|---|---:|---:|---:|---:|\']\n    for r in forward:report.append(f"| {r[\'model\']} | {r[\'batch_size\']} | {r[\'median_batch_ms\']:.3f} | {r[\'p95_batch_ms\']:.3f} | {r[\'throughput_images_per_second\']:.1f} |")\n    for name in [\'resnet18\',\'ensemble_50_50\']:\n        times=[r[\'seconds\'] for r in e2e if r[\'model\']==name]\n        report.append(f"\\n{name}: median end-to-end trên 1.028 ảnh, hai lượt đo thứ tự luân phiên = {np.median(times):.3f} giây ({1000*np.median(times)/1028:.3f} ms/ảnh theo thông lượng batch32).")\n    report += [\'\',f"model.pkl: {size[\'model_pkl_bytes\']:,} byte = {size[\'model_pkl_MiB\']:.2f} MiB. Tổng {size[\'ensemble_parameters\']:,} tham số. Forward-only không gồm đọc ảnh/tiền xử lý. End-to-end có đọc ảnh/tiền xử lý, warm cache, batch32, workers2; ms/ảnh ở đây là thời gian trung bình theo batch, không phải độ trễ ảnh đơn. Batch1 mới phản ánh forward ảnh đơn.",\n      \'\', \'## Cách dùng và lưu file\',\'\',\n      \'Giải nén vào PneumoScore-XAI/Outputs/Ensemble_Seed2026_Verified/. Giữ toàn bộ thư mục và model/resume cũ. Có thể dọn bản trùng trong Downloads sau khi kiểm tra bản đã lưu.\',\n      \'\', \'- model.pkl: trọng số hai model và cấu hình ghép/ngưỡng, cần giữ.\',\n      \'- submission.ipynb: mở trên Kaggle, Add Input model.pkl và Test BTC, điền TEST_DIR/mã đội/CSV mẫu, Run All để tự sinh CSV và ZIP bài nộp. Không sửa CSV bằng tay.\',\n      \'- verify_export.ipynb: tái lập phiên kiểm tra trên Kaggle T4; cần Output Stability, Output Ensemble Analysis và dataset BTC đã dùng.\',\n      \'- ensemble_inference.py: mã trong notebook để tham khảo và giải thích, không cần chạy riêng trên Windows.\',\n      \'- validation_*.csv, validation_metrics.csv, verification.json: bằng chứng suy luận lại và kiểm tra định dạng, không nộp làm kết quả Test.\',\n      \'- benchmark*.csv/json, environment.json, source_checkpoints.json và báo cáo này: lưu hồ sơ nghiên cứu.\',\n      \'\', \'Gói hiện tại là bộ model đã kiểm tra, chưa phải ZIP bài nộp vì chưa có ảnh Test BTC trong phiên này. Khi có Test, notebook sinh một .pkl, một .ipynb và team01.csv theo Điều 6. Không dùng nhãn Test.\',\n      \'\', \'P của cặp seed 2026 không phải trung bình ba seed. Các chỉ số vẫn là validation đã dùng chọn checkpoint/ensemble/ngưỡng; không chứng minh hiệu quả trên bệnh nhân mới. Giữ baseline làm phương án chính đã chuẩn hóa và ensemble làm ứng viên đối chứng.\']\n    (out/\'README_VI.md\').write_text(\'\\n\'.join(report)+\'\\n\',encoding=\'utf-8\')\n    dest=out.parent/\'PneumoScore_Ensemble_Seed2026_Verified.zip\'\n    with zipfile.ZipFile(dest,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(out.iterdir()):\n            if p.is_file():z.write(p,\'Ensemble_Seed2026_Verified/\'+p.name)\n    print(\'VALIDATION_METRICS\\n\'+pd.DataFrame(metric_rows).to_string(index=False),flush=True)\n    print(\'BENCHMARK_FORWARD\\n\'+pd.DataFrame(forward).to_string(index=False),flush=True)\n    print(\'BENCHMARK_E2E\\n\'+pd.DataFrame(e2e).to_string(index=False),flush=True)\n    print(\'MODEL_SIZE\',json.dumps(size),flush=True)\n    print(\'ENSEMBLE_EXPORT_VERIFIED_DONE\',str(dest),flush=True)\n    return dest\n'
m=types.ModuleType("ensemble_patch")
exec(compile(MODULE_SOURCE,"ensemble_inference.py","exec"),m.__dict__)


In [ ]:
from pathlib import Path
import json,io,zipfile,tempfile,shutil,subprocess,sys,gc,pickle,hashlib,time
import numpy as np
import pandas as pd
from PIL import Image
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score,log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def run_patch_verification(m,module_source):
    started=time.perf_counter();m.require_kaggle(t4=True)
    out=Path('/kaggle/working/ensemble_patch_verification');out.mkdir(exist_ok=True)
    source=m.resolve_model({});m.ensure(m.file_hash(source)==m.LEGACY_MODEL_SHA256,'Attach the exact old Export Verify output.')
    manifests=[]
    for p in Path('/kaggle/input').rglob('split_manifest.csv'):
        f=pd.read_csv(p,keep_default_na=False)
        if set(['relative_path','label','split','sha256','pixel_sha256','patient_id','group_id']).issubset(f.columns) and m.manifest_fingerprint(f)=='9f9474fafb054f501535269fa3e55bea4c3e24d794f6afe9c019abe6ce798ac2':manifests.append((p,f))
    m.ensure(manifests,'Attach Stability 3 Seeds output.')
    manifest_path,manifest=manifests[0];root=m.resolve_data_root({},manifest)
    val=manifest[manifest.split=='validation'].sort_values('image_id').reset_index(drop=True)
    refs=list(Path('/kaggle/input').rglob('validation_predictions_reloaded.csv'));m.ensure(len(refs)==1,'Expected one verified prediction reference.')
    reference=pd.read_csv(refs[0]).sort_values('image_id').reset_index(drop=True)
    m.ensure(reference.image_id.tolist()==val.image_id.tolist(),'Reference IDs mismatch.')
    cfg={'BATCH_SIZE':32,'NUM_WORKERS':2}
    device=m.torch.device('cuda:0')
    net,payload=m.load_pickle(source,device)
    actual,_=m.predict(net,m.loader_for(val,cfg,root),device,True)
    raw_error=float(np.max(np.abs(actual.p_pneumonia.to_numpy()-reference.ensemble_probability.to_numpy())))
    m.ensure(np.allclose(actual.p_pneumonia,reference.ensemble_probability,atol=1e-6,rtol=1e-5),'Original-model probability regression.')
    m.ensure(np.array_equal(actual.p_pneumonia>=.55,reference.ensemble_probability>=.55),'Original labels changed.')
    print('ORIGINAL_1028_RELOAD_OK max_error',raw_error,flush=True)
    result=dict(status='running',source_model_sha256=m.file_hash(source),architecture_weights_threshold_unchanged=True,
        original_validation=dict(n=len(val),max_probability_error=raw_error,changed_labels=0,metrics=m.metrics(val.label,actual.p_pneumonia,.55)),tests={})
    with tempfile.TemporaryDirectory(prefix='pneumoscore_patch_smoke_',dir='/kaggle/working') as temp:
        tmp=Path(temp);portable=tmp/'portable.pkl';decoded=m.save_portable_pickle(payload,portable,m.file_hash(source))
        result['portable_model_sha256']=m.file_hash(portable)
        result['tests']['tensor_bytes_exactly_preserved']=True
        # The portable reader has no class/global imports; NumPy byte reconstruction occurs afterwards.
        with portable.open('rb') as f:raw=m.BuiltinsOnlyUnpickler(f).load()
        result['tests']['portable_builtins_only_load']=raw['export_schema']=='pneumoscore_probability_ensemble_bytes_v2'
        del net;gc.collect();m.torch.cuda.empty_cache()
        net2,payload2=m.load_pickle(portable,device)
        actual2,_=m.predict(net2,m.loader_for(val,cfg,root),device,True)
        result['portable_validation']=dict(n=len(val),max_probability_error=float(np.max(np.abs(actual2.p_pneumonia-actual.p_pneumonia))),changed_labels=int(((actual2.p_pneumonia>=.55)!=(actual.p_pneumonia>=.55)).sum()),metrics=m.metrics(val.label,actual2.p_pneumonia,.55))
        m.ensure(np.array_equal(actual2.p_pneumonia.to_numpy(),actual.p_pneumonia.to_numpy()),'Portable inference differs from original.')
        del net2;gc.collect();m.torch.cuda.empty_cache()
        fixture=tmp/'fixture';fixture.mkdir()
        # Copies of four existing BTC Validation images, anonymous names. These are NOT official Test images.
        chosen=val.iloc[[0,1,2,3]]
        for index,row in enumerate(chosen.itertuples(index=False),1):
            im=m.read_xray(root/row.relative_path);im.save(fixture/f'CASE_{index:03d}.png')
        run_cfg=dict(MODE='smoke',MODEL_PATH=str(source),TEST_DIR=str(fixture),SAMPLE_SUBMISSION_CSV='',TEAM_ID='smoke01',BATCH_SIZE=4,NUM_WORKERS=0,OUTPUT=str(tmp/'run_no_sample'),_MODULE_SOURCE=module_source)
        first=m.run_submission(run_cfg);first_csv=pd.read_csv(first.parent/'smoke01.csv',dtype=str)
        first_meta=json.loads((first.parent/'submission_metadata.json').read_text())
        m.ensure(len(first_csv)==4 and first_meta['overlap_images']==4 and first_meta['eligibility_review_required'],'Overlap should warn and retain all fixture predictions.')
        m.ensure(set(first_csv.Prediction).issubset({'Normal','Pneumonia'}),'Wrong external label format.')
        result['tests']['end_to_end_no_sample_overlap_retained']=True
        result['tests']['external_labels_BTC_strings']=True
        sample=tmp/'sample.csv';ids=['case_004','CASE_002.PNG','case_001.png','CASE_003']
        pd.DataFrame({'Image_id':ids,'Prediction':['UNREAD_SAMPLE_VALUE']*4}).to_csv(sample,index=False)
        with_sample=dict(run_cfg,SAMPLE_SUBMISSION_CSV=str(sample),OUTPUT=str(tmp/'run_with_sample'))
        second=m.run_submission(with_sample);second_csv=pd.read_csv(second.parent/'smoke01.csv',dtype=str)
        m.ensure(second_csv.Image_id.tolist()==ids,'Authoritative sample ID spelling/order changed.')
        mapping=pd.read_csv(second.parent/'test_id_mapping.csv')
        result['tests']['sample_exact_case_stem_order']=True
        result['tests']['sample_predictions_not_used']=True
        result['id_mapping_example']=mapping.to_dict('records')
        # Execute the exact emitted notebook from its ZIP in a new Python process on Kaggle.
        replay=tmp/'replay';replay.mkdir()
        with zipfile.ZipFile(second) as z:
            m.ensure(z.testzip() is None,'Smoke ZIP CRC failed.')
            names=z.namelist();m.ensure({'model.pkl','smoke01.csv','submission.ipynb'}.issubset(names),'Required ZIP files missing.')
            z.extractall(replay)
        harness=tmp/'execute_notebook.py'
        harness.write_text("import json,sys\nfrom pathlib import Path\nnb=json.loads(Path(sys.argv[1]).read_text())\nns={'__name__':'__main__'}\nfor i,c in enumerate(nb['cells']):\n if c['cell_type']=='code':exec(compile(''.join(c['source']),'replay_cell_'+str(i),'exec'),ns)\n")
        completed=subprocess.run([sys.executable,'-O',str(harness),str(replay/'submission.ipynb')],capture_output=True,text=True,timeout=180)
        m.ensure(completed.returncode==0,'Emitted notebook replay failed: '+completed.stderr[-2000:])
        replay_csv=pd.read_csv(second.parent/'smoke01.csv',dtype=str)
        m.ensure(replay_csv.equals(second_csv),'Emitted notebook did not reproduce the same CSV.')
        result['tests']['emitted_ZIP_notebook_fresh_process_replays_identical_CSV']=True
        result['tests']['critical_validation_survives_python_optimized_mode']=True
        result['tests']['smoke_zip_CRC_and_required_files']=True
        result['smoke_zip_entries']=names
        # Run critical validation under Python -O, not only normal assertion-enabled Python.
        malicious=tmp/'forbidden.pkl';malicious.write_bytes(pickle.dumps(eval,protocol=4))
        try:
            with malicious.open('rb') as f:m.BuiltinsOnlyUnpickler(f).load()
        except pickle.UnpicklingError:result['tests']['portable_global_import_rejected']=True
        else:raise ValueError('Restricted loader accepted a global.')
        invalid=fixture/'BROKEN.png';invalid.write_bytes(b'not an image')
        Image.fromarray(np.arange(64,dtype=np.uint16).reshape(8,8)).save(fixture/'SIXTEEN_BIT.png')
        invalid_cfg=dict(run_cfg,OUTPUT=str(tmp/'invalid_images'))
        try:m.run_submission(invalid_cfg)
        except ValueError:
            errors=pd.read_csv(Path(invalid_cfg['OUTPUT'])/'input_errors.csv')
            m.ensure(len(errors)==2 and not (Path(invalid_cfg['OUTPUT'])/'smoke01.csv').exists(),'Bad images must be reported together without fabricated predictions.')
            result['tests']['corrupt_and_16bit_reported_no_guessed_predictions']=True
        else:raise ValueError('Invalid images unexpectedly accepted.')
        invalid.unlink();(fixture/'SIXTEEN_BIT.png').unlink()
        duplicate=fixture/'sub';duplicate.mkdir();shutil.copy2(fixture/'CASE_001.png',duplicate/'CASE_001.png')
        try:m.collect_test(run_cfg,payload,tmp)
        except ValueError:result['tests']['ambiguous_duplicate_basename_rejected']=True
        else:raise ValueError('Ambiguous basename accepted.')
        shutil.rmtree(duplicate)
        pd.DataFrame({'Image_id':['MISSING_IMAGE.png']}).to_csv(sample,index=False)
        try:m.collect_test(with_sample,payload,tmp)
        except ValueError:result['tests']['missing_sample_ID_rejected']=True
        else:raise ValueError('Missing sample ID accepted.')
        (fixture/'UNSUPPORTED.dcm').write_bytes(b'unsupported DICOM fixture')
        try:m.collect_test(run_cfg,payload,tmp)
        except ValueError:
            m.ensure('Unsupported image format' in (tmp/'input_errors.csv').read_text(),'Unsupported format not logged.')
            result['tests']['unsupported_DICOM_reported']=True
        else:raise ValueError('Unsupported DICOM silently accepted.')
        (fixture/'UNSUPPORTED.dcm').unlink()
        if '.webp' in Image.registered_extensions():
            im=m.read_xray(fixture/'CASE_001.png');im.save(fixture/'WEBP_CASE.webp',lossless=True)
            frame=m.collect_test(run_cfg,payload,tmp);m.ensure(len(frame)==5,'WebP file skipped.')
            result['tests']['webp_scanned_and_read']=True
        # Old output is archived before a failed rerun, preventing a stale final CSV/ZIP.
        (fixture/'BROKEN.png').write_bytes(b'bad')
        try:m.run_submission(run_cfg)
        except ValueError:
            m.ensure(not first.exists() and not (first.parent/'smoke01.csv').exists(),'Stale output remains under final name.')
            result['tests']['failed_rerun_archives_previous_final_artifacts']=True
        else:raise ValueError('Broken-image rerun unexpectedly passed.')
        result['tests']['submit_mode_rejects_working_fixture_root']=False
        try:m.collect_test(dict(run_cfg,MODE='submit'),payload,tmp)
        except ValueError:result['tests']['submit_mode_rejects_working_fixture_root']=True
        m.ensure(result['tests']['submit_mode_rejects_working_fixture_root'],'Smoke fixture could be treated as official Test.')
    # Review audit using saved probabilities: no new threshold/model is adopted.
    y=reference.label.to_numpy(dtype=int)
    probs={'baseline':reference.baseline_probability.to_numpy(),'ECA':reference.eca_probability.to_numpy(),'ensemble':reference.ensemble_probability.to_numpy()}
    locked={'baseline':.35,'ECA':.5,'ensemble':.55};statistics={'fixed_checkpoint_conditional_only':True,'mcnemar':{},'threshold_grid_001':{}}
    ts=np.arange(.05,.951,.01)
    for name,p in probs.items():
        sweep=pd.DataFrame([dict(model=name,**m.metrics(y,p,float(t))) for t in ts]);sweep.to_csv(out/(name+'_posthoc_sweep_001.csv'),index=False)
        best=sweep.loc[sweep.P.idxmax()];statistics['threshold_grid_001'][name]={'threshold':float(best.threshold),'P':float(best.P),'AUC':float(roc_auc_score(y,p)),'BCE':float(log_loss(y,p))}
    for name in ['baseline','ECA']:
        a=(probs['ensemble']>=.55)==y;b=(probs[name]>=locked[name])==y
        counts={'only_ensemble_correct':int((a&~b).sum()),'only_other_correct':int((~a&b).sum()),'both_correct':int((a&b).sum()),'both_wrong':int((~a&~b).sum())}
        n=counts['only_ensemble_correct']+counts['only_other_correct'];counts['p_exact']=float(binomtest(counts['only_ensemble_correct'],n,.5).pvalue) if n else 1.
        counts['P_delta_pp']=100*(m.metrics(y,probs['ensemble'],.55)['P']-m.metrics(y,probs[name],locked[name])['P']);statistics['mcnemar'][name]=counts
    rng=np.random.default_rng(2026);neg=np.flatnonzero(y==0);pos=np.flatnonzero(y==1);deltas=[]
    for _ in range(10000):
        idx=np.concatenate([rng.choice(neg,len(neg),replace=True),rng.choice(pos,len(pos),replace=True)])
        deltas.append(100*(m.metrics(y[idx],probs['ensemble'][idx],.55)['P']-m.metrics(y[idx],probs['baseline'][idx],.35)['P']))
    statistics['image_stratified_bootstrap_ens_minus_base']={'B':10000,'rng_seed':2026,'ci95_pp':np.quantile(deltas,[.025,.975]).tolist(),'scope':'paired class-stratified images; fixed checkpoints/thresholds; not patient-independent and not selection-adjusted'}
    def p_scores(labels,p,t):
        pred=p[None,:]>=t[:,None];positive=labels[None,:]==1
        tp=(pred&positive).sum(1);fp=(pred&~positive).sum(1);fn=positive.sum()-tp;tn=(~positive).sum()-fp
        return .3*(tp+tn)/len(labels)+.25*tp/(tp+fn)+.2*tn/(tn+fp)+.25*2*tp/(2*tp+fp+fn)
    optimistic={name:[] for name in probs};held={name:[] for name in probs};selected={name:[] for name in probs}
    for _ in range(2000):
        n=rng.permutation(neg);p=rng.permutation(pos);a=np.r_[n[:len(n)//2],p[:len(p)//2]];b=np.r_[n[len(n)//2:],p[len(p)//2:]]
        for name,prob in probs.items():
            pa=p_scores(y[a],prob[a],ts);k=int(np.argmax(pa));pb=float(p_scores(y[b],prob[b],ts[k:k+1])[0])
            optimistic[name].append(100*(pa[k]-pb));held[name].append(100*pb);selected[name].append(float(ts[k]))
    statistics['split_half_threshold_diagnostic']={name:{'B':2000,'mean_tune_minus_other_half_pp':float(np.mean(optimistic[name])),'mean_other_half_P_percent':float(np.mean(held[name])),'threshold_median':float(np.median(selected[name])),'threshold_IQR':np.quantile(selected[name],[.25,.75]).tolist()} for name in probs}
    statistics['split_half_limitation']='Both halves already influenced original checkpoint/model selection; measures an additional threshold-selection sensitivity at half sample size, not the exact optimism of the final full-validation score or an independent Test estimate.'
    train=manifest[manifest.split=='train'];geometry={}
    for name,features in {'aspect_ratio':lambda f:np.c_[f.width.to_numpy()/f.height.to_numpy()], 'width_height':lambda f:np.c_[f.width.to_numpy(),f.height.to_numpy()]}.items():
        clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=2026));clf.fit(features(train),train.label)
        p=clf.predict_proba(features(val))[:,1];geometry[name]={'validation_AUC':float(roc_auc_score(val.label,p)),'P_at_05':m.metrics(val.label,p,.5)['P']}
    statistics['geometry_association']=geometry;statistics['geometry_interpretation']='Association between image geometry and labels, not proof that the CNN uses a shortcut, and not a forecast of BTC Test performance.'
    statistics['random_image_split_neighbor_probability_if_4_neighbors']=1-.2**4
    m.save_json(statistics,out/'review_statistics.json')
    result['status']='passed';result['seconds']=time.perf_counter()-started;result['environment']=m.execution_environment()
    result['official_BTC_Test_executed']=False;result['no_training']=True;result['threshold_not_retuned']=True
    m.save_json(result,out/'patch_verification.json')
    (out/'ensemble_inference.py').write_text(module_source,encoding='utf-8')
    m.save_json(m.make_submission_notebook(module_source),out/'submission.ipynb')
    print('PATCH_VERIFICATION_PASSED',json.dumps(result,ensure_ascii=False),flush=True)
    print('REVIEW_STATISTICS',json.dumps(statistics,ensure_ascii=False),flush=True)
    dest=Path('/kaggle/working/PneumoScore_Ensemble_Patch_Verification.zip')
    with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out.iterdir()):z.write(p,p.name)
    print('VERIFICATION_REPORT_ZIP',dest,flush=True)
    return dest

run_patch_verification(m,MODULE_SOURCE)
